# Notebook 02: Discount Analysis
**Objective:** Analyse discount depth by segment and rep, identify over-discounters, and quantify the revenue impact of discounting across the portfolio.

In [ ]:
import pandas as pd
import numpy as np
import duckdb
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
deals_df = pd.read_csv('../data/raw/deals.csv')
outcomes_df = pd.read_csv('../data/raw/outcomes.csv')

deals_df['created_date'] = pd.to_datetime(deals_df['created_date'])
outcomes_df['close_date'] = pd.to_datetime(outcomes_df['close_date'])

con = duckdb.connect()
con.register('deals', deals_df)
con.register('outcomes', outcomes_df)

merged = deals_df.merge(outcomes_df, on='deal_id')

print(f"deals: {len(deals_df):,} rows")
print(f"outcomes: {len(outcomes_df):,} rows")
print(f"merged: {len(merged):,} rows")

## Discount Distribution by Segment

In [ ]:
segments = ['SMB', 'Mid-Market', 'Enterprise']
colors = ['#636EFA', '#EF553B', '#00CC96']

fig = go.Figure()

for seg, color in zip(segments, colors):
    d = merged[merged['segment'] == seg]['discount_pct']
    fig.add_trace(go.Box(
        y=d,
        name=seg,
        marker_color=color,
        boxmean=True
    ))

fig.update_layout(
    title='Discount Distribution by Segment',
    yaxis_title='Discount %',
    yaxis=dict(tickformat='.0%', range=[0, 0.55]),
    height=500,
    font=dict(size=13, color='#212121'),
    template='plotly_white'
)

fig.show()
fig.write_image('../outputs/01_discount_by_segment.png')

## Rep-Level Discount Analysis

In [ ]:
deals_df = pd.read_csv('../data/raw/deals.csv')
outcomes_df = pd.read_csv('../data/raw/outcomes.csv')

deals_df['created_date'] = pd.to_datetime(deals_df['created_date'])
outcomes_df['close_date'] = pd.to_datetime(outcomes_df['close_date'])

con = duckdb.connect()
con.register('deals', deals_df)
con.register('outcomes', outcomes_df)

merged = deals_df.merge(outcomes_df, on='deal_id')

print(f"deals: {len(deals_df):,} rows")
print(f"outcomes: {len(outcomes_df):,} rows")
print(f"merged: {len(merged):,} rows")

In [ ]:
OVERDISCOUNT_THRESHOLD = 0.15

for deal_type in ['New Business', 'Expansion', 'Renewal']:
    rep_dt = con.execute(f"""
        SELECT
            d.rep_id,
            COUNT(*) AS total_deals,
            ROUND(AVG(d.discount_pct), 4) AS avg_discount,
            ROUND(MAX(d.discount_pct), 4) AS max_discount,
            ROUND(AVG(o.win_flag), 4) AS win_rate
        FROM deals d
        JOIN outcomes o ON d.deal_id = o.deal_id
        WHERE d.deal_type = '{deal_type}'
        GROUP BY d.rep_id
        HAVING COUNT(*) >= 5
        ORDER BY avg_discount DESC
    """).df()

    rep_dt['over_discounter'] = rep_dt['avg_discount'] > OVERDISCOUNT_THRESHOLD

    print(f"\n--- {deal_type} ---")
    print(
        f"{'Rep':<12} {'Deals':>8} {'Avg Disc':>10} "
        f"{'Max Disc':>10} {'Win Rate':>10} {'Flag':>6}"
    )
    print("-" * 58)
    for _, row in rep_dt.iterrows():
        flag = '⚠' if row['over_discounter'] else ''
        print(
            f"{row['rep_id']:<12} {row['total_deals']:>8,} "
            f"{row['avg_discount']:>9.1%} {row['max_discount']:>9.1%} "
            f"{row['win_rate']:>9.1%} {flag:>6}"
        )

## Rep Discount vs Win Rate — New Business Only

In [ ]:
nb_only = con.execute("""
    SELECT
        d.rep_id,
        COUNT(*) AS total_deals,
        ROUND(AVG(d.discount_pct), 4) AS avg_discount,
        ROUND(AVG(o.win_flag), 4) AS win_rate
    FROM deals d
    JOIN outcomes o ON d.deal_id = o.deal_id
    WHERE d.deal_type = 'New Business'
    GROUP BY d.rep_id
    HAVING COUNT(*) >= 5
    ORDER BY avg_discount DESC
""").df()

nb_only['over_discounter'] = nb_only['avg_discount'] > OVERDISCOUNT_THRESHOLD
colors = nb_only['over_discounter'].map({True: '#EF553B', False: '#636EFA'})

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=nb_only['avg_discount'],
    y=nb_only['win_rate'],
    mode='markers+text',
    text=nb_only['rep_id'],
    textposition='top center',
    marker=dict(size=10, color=colors),
))

fig.add_vline(
    x=OVERDISCOUNT_THRESHOLD,
    line_dash='dash',
    line_color='grey',
    annotation_text='15% threshold',
    annotation_position='top right'
)

fig.update_layout(
    title='Rep Avg Discount vs Win Rate — New Business Only (red = over-discounter)',
    xaxis_title='Avg Discount %',
    yaxis_title='Win Rate',
    xaxis=dict(tickformat='.0%', range=[0, 0.30]),
    yaxis=dict(tickformat='.0%', range=[0, 0.80]),
    height=550,
    font=dict(size=13, color='#212121'),
    template='plotly_white'
)

fig.show()
fig.write_image('../outputs/02_rep_discount_vs_winrate.png')

## Revenue Impact of Discounting

In [ ]:
revenue_impact = con.execute("""
    SELECT
        d.segment,
        COUNT(*) AS total_deals,
        ROUND(SUM(d.list_price), 0) AS total_list_value,
        ROUND(SUM(d.list_price * d.discount_pct), 0) AS total_discount_given,
        ROUND(
            SUM(d.list_price * d.discount_pct) / SUM(d.list_price), 4
        ) AS effective_discount_rate,
        ROUND(SUM(o.final_arr), 0) AS total_final_arr
    FROM deals d
    JOIN outcomes o ON d.deal_id = o.deal_id
    GROUP BY d.segment
    ORDER BY total_list_value DESC
""").df()

cols = ['total_list_value', 'total_discount_given', 'total_final_arr']

header = f"{'Segment':<15} {'Deals':>8} {'List Value':>14} {'Disc Given':>14} {'Eff Disc%':>10} {'Final ARR':>14}"
print(header)
print("-" * 77)

for _, row in revenue_impact.iterrows():
    print(
        f"{row['segment']:<15} {row['total_deals']:>8,} "
        f"${row['total_list_value']:>12,.0f} "
        f"${row['total_discount_given']:>12,.0f} "
        f"{row['effective_discount_rate']:>9.1%} "
        f"${row['total_final_arr']:>12,.0f}"
    )

total = revenue_impact[cols].sum()
eff_disc = total['total_discount_given'] / total['total_list_value']
print(
    f"\n{'Overall':<15} {revenue_impact['total_deals'].sum():>8,} "
    f"${total['total_list_value']:>12,.0f} "
    f"${total['total_discount_given']:>12,.0f} "
    f"{eff_disc:>9.1%} "
    f"${total['total_final_arr']:>12,.0f}"
)

## Discount Bands vs Win Rate

In [ ]:
bins = [0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.50]
labels = ['0-5%', '5-10%', '10-15%', '15-20%', '20-25%', '25%+']

merged['discount_band'] = pd.cut(
    merged['discount_pct'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

band_summary = (
    merged.groupby('discount_band', observed=True)
    .agg(
        deals=('deal_id', 'count'),
        win_rate_count=('win_flag', 'mean'),
        avg_discount=('discount_pct', 'mean'),
        total_arr=('arr', 'sum'),
        won_arr=('final_arr', 'sum')
    )
    .reset_index()
)

band_summary['win_rate_value'] = (
    band_summary['won_arr'] / band_summary['total_arr']
)

print(
    f"{'Band':<12} {'Deals':>8} {'Avg Disc':>10} "
    f"{'WR (Count)':>12} {'WR (Value)':>12}"
)
print("-" * 56)
for _, row in band_summary.iterrows():
    print(
        f"{str(row['discount_band']):<12} {row['deals']:>8,} "
        f"{row['avg_discount']:>9.1%} {row['win_rate_count']:>11.1%} "
        f"{row['win_rate_value']:>11.1%}"
    )

## Discount Band vs Win Rate Chart

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=band_summary['discount_band'].astype(str),
    y=band_summary['win_rate_count'],
    mode='lines+markers',
    name='Win Rate (Count)',
    line=dict(color='#636EFA', width=2),
    marker=dict(size=8)
))

fig.add_trace(go.Scatter(
    x=band_summary['discount_band'].astype(str),
    y=band_summary['win_rate_value'],
    mode='lines+markers',
    name='Win Rate (Value)',
    line=dict(color='#EF553B', width=2),
    marker=dict(size=8)
))

fig.update_layout(
    title='Win Rate by Discount Band (Count vs Value)',
    xaxis_title='Discount Band',
    yaxis_title='Win Rate',
    yaxis=dict(tickformat='.0%', range=[0, 0.70]),
    height=500,
    font=dict(size=13, color='#212121'),
    template='plotly_white',
    legend=dict(x=0.75, y=0.95)
)

fig.show()
fig.write_image('../outputs/03_winrate_by_discount_band.png')

## Discount Given: Won vs Lost Deals

In [ ]:
discount_split = con.execute("""
    SELECT
        o.stage,
        COUNT(*) AS total_deals,
        ROUND(AVG(d.discount_pct), 4) AS avg_discount,
        ROUND(SUM(d.list_price * d.discount_pct), 0) AS discount_given,
        ROUND(SUM(d.list_price), 0) AS total_list_value
    FROM deals d
    JOIN outcomes o ON d.deal_id = o.deal_id
    GROUP BY o.stage
    ORDER BY o.stage DESC
""").df()

discount_split['pct_of_total'] = (
    discount_split['discount_given'] / discount_split['discount_given'].sum()
)

print(
    f"{'Stage':<15} {'Deals':>8} {'Avg Disc':>10} "
    f"{'Disc Given':>14} {'% of Total':>12}"
)
print("-" * 61)
for _, row in discount_split.iterrows():
    print(
        f"{row['stage']:<15} {row['total_deals']:>8,} "
        f"{row['avg_discount']:>9.1%} "
        f"${row['discount_given']:>12,.0f} "
        f"{row['pct_of_total']:>11.1%}"
    )

## Discount by Deal Type

In [ ]:
deal_type_summary = con.execute("""
    SELECT
        d.deal_type,
        COUNT(*) AS total_deals,
        ROUND(AVG(d.discount_pct), 4) AS avg_discount,
        ROUND(AVG(o.win_flag), 4) AS win_rate_count,
        ROUND(SUM(d.list_price * d.discount_pct), 0) AS discount_given,
        ROUND(
            SUM(o.final_arr) / NULLIF(SUM(d.list_price), 0), 4
        ) AS win_rate_value
    FROM deals d
    JOIN outcomes o ON d.deal_id = o.deal_id
    GROUP BY d.deal_type
    ORDER BY avg_discount DESC
""").df()

print(
    f"{'Deal Type':<16} {'Deals':>8} {'Avg Disc':>10} "
    f"{'WR (Count)':>12} {'WR (Value)':>12} {'Disc Given':>14}"
)
print("-" * 74)
for _, row in deal_type_summary.iterrows():
    print(
        f"{row['deal_type']:<16} {row['total_deals']:>8,} "
        f"{row['avg_discount']:>9.1%} {row['win_rate_count']:>11.1%} "
        f"{row['win_rate_value']:>11.1%} ${row['discount_given']:>12,.0f}"
    )

## Win Rate by Deal Type

In [ ]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=deal_type_summary['deal_type'],
    y=deal_type_summary['win_rate_count'],
    name='Win Rate (Count)',
    text=[f"{v:.1%}" for v in deal_type_summary['win_rate_count']],
    textposition='outside',
    marker_color='#636EFA'
))

fig.add_trace(go.Bar(
    x=deal_type_summary['deal_type'],
    y=deal_type_summary['win_rate_value'],
    name='Win Rate (Value)',
    text=[f"{v:.1%}" for v in deal_type_summary['win_rate_value']],
    textposition='outside',
    marker_color='#EF553B'
))

fig.update_layout(
    title='Win Rate by Deal Type (Count vs Value)',
    xaxis_title='Deal Type',
    yaxis_title='Win Rate',
    yaxis=dict(tickformat='.0%', range=[0, 0.90]),
    barmode='group',
    height=500,
    font=dict(size=13, color='#212121'),
    template='plotly_white',
    legend=dict(x=0.75, y=0.95)
)

fig.show()
fig.write_image('../outputs/04_winrate_by_deal_type.png')

## Key Findings

- Overall avg discount is 11.2%; Enterprise deals average 14.6%
- $37.1M in discounts given across the portfolio on $274M list value (13.5% effective rate)
- 54.1% of all discount given ($20.1M) was on lost deals — pure margin waste
- Win rate (count) peaks at 5-10% discount band (63.4%) and drops sharply above 20% (46.7%)
- Value-based win rate is consistently lower than count — large deals are lost more often than small ones
- The 15-20% band shows the greatest divergence — reps discount heavily on large deals but still lose them
- Renewals win 71.8% of deals at 9.3% avg discount; New Business wins 47.9% at 12.4%
- New Business accounts for $20.3M of discount given — anchoring initial contract value matters most
- 4 reps exceed the 15% threshold on New Business deals; they win at 44-53% — no better than disciplined reps
- REP_010 wins 61.4% of New Business deals at 11.6% avg discount — most efficient new logo closer
- REP_020 anomaly: 10.4% avg discount on New Business but only 25.4% win rate — needs investigation
- REP_017 wins 93.8% of renewals at 6.0% avg discount — best renewal rep in the team